# 02 — AI 1 Retrieval Test & Evidence

**Mục tiêu:** chạy các `UserPreferenceState` thực tế qua `Retriever`, kiểm tra hard constraints và AI 3 contract, đo latency/coverage, lưu top candidates để đánh giá thủ công. **Không tự nhận precision/recall** nếu chưa có nhãn relevance. Notebook này cần Parquet + embeddings từ Phần 1–3 và source `src/retrieval/` Phần 2–5.

Chạy từ gốc project `mlai-food` (`jupyter lab`) hoặc mở trong VS Code.

In [ ]:
%pip install sentence-transformers

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT = Path.cwd()

PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports" / "ai1"
REPORTS.mkdir(parents=True, exist_ok=True)
restaurants = pd.read_parquet(PROCESSED / "restaurants.parquet")
dishes = pd.read_parquet(PROCESSED / "dishes.parquet")
menus = pd.read_parquet(PROCESSED / "menu_items.parquet")
print('ROOT:', ROOT)
print('Rows:', {"restaurants":len(restaurants), "dishes":len(dishes), "menu_items":len(menus)})

In [ ]:
import sys, time
if str(ROOT / "src") not in sys.path: sys.path.insert(0, str(ROOT / "src"))
from common.schemas import Candidate, Location, UserPreferenceState
from retrieval import retrieve_candidates
from retrieval.retriever import Retriever
EMBEDDINGS=ROOT/'data'/'embeddings'
assert (EMBEDDINGS/'metadata.json').exists(), 'Chạy offline embedding trước: data/embeddings/metadata.json chưa có'
meta=json.loads((EMBEDDINGS/'metadata.json').read_text(encoding='utf-8'))
print('Model:',meta.get('model'),'Version:',meta.get('dataset_version'))
t0=time.perf_counter();engine=Retriever(processed_dir=PROCESSED,embeddings_dir=EMBEDDINGS)
print('Engine load seconds:',round(time.perf_counter()-t0,2))


## 1. Truy vấn mẫu và contract AI 3

In [ ]:
cases = [
    {'name':'bun_chay_hanoi','state':UserPreferenceState(session_id='bun-chay', location=Location(lat=21.03,lon=105.85), food=['bún chay'], cuisine=['vietnamese'], hard_constraints={'budget_max':100000,'max_distance_km':5,'vegetarian':True}, soft_preferences={'ít dầu':True}, semantic_preferences=['thanh đạm'])},
    {'name':'pho_hanoi','state':UserPreferenceState(session_id='pho', location=Location(lat=21.03,lon=105.85), food=['phở'], cuisine=['vietnamese'], hard_constraints={'budget_max':80000,'max_distance_km':5})},
    {'name':'vegetarian_nearby','state':UserPreferenceState(session_id='veg', location=Location(lat=21.03,lon=105.85), food=['món chay'], hard_constraints={'budget_max':60000,'max_distance_km':3,'vegetarian':True})},
    {'name':'no_semantic_text','state':UserPreferenceState(session_id='fallback', location=Location(lat=21.03,lon=105.85), hard_constraints={'budget_max':100000,'max_distance_km':5})},
]
results={}; candidate_results={}; stats=[]
for case in cases:
    state=case['state']; name=case['name']; t=time.perf_counter()
    candidates=retrieve_candidates(state,50,retriever=engine)
    elapsed=time.perf_counter()-t
    assert len(candidates)<=50 and all(isinstance(c,Candidate) for c in candidates)
    ids=[c.menu_item_id for c in candidates]; assert len(ids)==len(set(ids))
    budget=state.hard_constraints.get('budget_max'); radius=state.hard_constraints.get('max_distance_km')
    if budget is not None: assert all(c.price is not None and c.price<=budget for c in candidates)
    if radius is not None: assert all(c.distance_km is not None and c.distance_km<=radius+1e-6 for c in candidates)
    if state.food or state.cuisine or state.soft_preferences or state.semantic_preferences:
        assert all('semantic_similarity' in c.match_features for c in candidates)
    candidate_results[name]=candidates
    # Internal frame is retained only for AI1 manual relevance review.
    internal={'location':[state.location.lat,state.location.lon] if state.location else None,'hard_constraints':dict(state.hard_constraints),'food':state.food,'cuisine':state.cuisine,'soft_preferences':state.soft_preferences,'semantic_preferences':state.semantic_preferences}
    frame=engine.retrieve_frame(internal,50); results[name]=frame
    stats.append({'case':name,'returned':len(candidates),'unique_restaurants':len({c.restaurant_id for c in candidates}),'latency_s':round(elapsed,3),'max_price':max((c.price for c in candidates if c.price is not None),default=None),'max_distance_km':max((c.distance_km for c in candidates if c.distance_km is not None),default=None),'contract_ok':True,'hard_filters_ok':True})
    print('PASS:',name)
display(pd.DataFrame(stats))


## 2. Kiểm tra kết quả có dễ giải thích không?

In [ ]:
name = 'bun_chay_hanoi'
frame = results[name].copy()

print("Frame columns:", frame.columns.tolist())

# Nếu retrieval output đã có dish_id thì dùng trực tiếp.
# Nếu không có mới lookup từ menu_items.
if 'dish_id' not in frame.columns:
    frame = frame.merge(
        menus[['menu_item_id', 'dish_id']],
        on='menu_item_id',
        how='left',
        validate='one_to_one'
    )

review = (
    frame
    .merge(
        dishes[
            [
                'dish_id',
                'dish_name',
                'supported_cuisine',
                'cuisine_confidence',
                'is_vegetarian'
            ]
        ],
        on='dish_id',
        how='left',
        validate='many_to_one'
    )
    .merge(
        restaurants[
            ['food_place_id', 'name']
        ],
        left_on='restaurant_id',
        right_on='food_place_id',
        how='left',
        validate='many_to_one'
    )
)

display(
    review[
        [
            'restaurant_id',
            'name',
            'dish_name',
            'supported_cuisine',
            'cuisine_confidence',
            'is_vegetarian',
            'distance_km',
            'price',
            'semantic_score'
        ]
    ].head(20)
)

print(
    "CẢNH BÁO: Điểm cosine cao không chứng minh món liên quan; "
    "cần gán nhãn thủ công."
)

## 3. Tạo mẫu đánh giá thủ công (relevance 0/1/2)

Gán `relevance`: **0** không liên quan; **1** có liên quan một phần; **2** phù hợp rõ ràng. Không dùng semantic score làm ground truth. Review ít nhất 20 kết quả mỗi truy vấn, có thể thêm truy vấn thật của team.

In [ ]:
samples = []

for case in cases:
    frame = results[case['name']].head(20).copy()

    # Chỉ lookup dish_id nếu Retriever chưa trả dish_id
    if 'dish_id' not in frame.columns:
        frame = frame.merge(
            menus[['menu_item_id', 'dish_id']],
            on='menu_item_id',
            how='left',
            validate='one_to_one'
        )

    # Thêm tên món
    named = (
        frame
        .merge(
            dishes[['dish_id', 'dish_name']],
            on='dish_id',
            how='left',
            validate='many_to_one'
        )
        .merge(
            restaurants[['food_place_id', 'name']],
            left_on='restaurant_id',
            right_on='food_place_id',
            how='left',
            validate='many_to_one'
        )
    )

    for row in named.to_dict('records'):
        samples.append({
            'query_id': case['name'],
            'query_text': str(case['state']),
            'menu_item_id': row['menu_item_id'],
            'restaurant_name': row.get('name'),
            'dish_name': row.get('dish_name'),
            'semantic_score': row['semantic_score'],

            # Điền thủ công sau
            'relevance': '',
            'review_notes': ''
        })


review_path = REPORTS / 'retrieval_manual_review.csv'

if review_path.exists():
    print('Existing review NOT overwritten:', review_path)
else:
    review_df = pd.DataFrame(samples)

    review_df.to_csv(
        review_path,
        index=False,
        encoding='utf-8-sig'
    )

    print(
        'Review template:',
        review_path,
        'rows:',
        len(review_df)
    )

    display(review_df.head(20))

## 4. Tính Precision@K / nDCG@K khi có nhãn

Sau khi điền `relevance` trong CSV, chạy lại cell dưới đây. Nếu chưa có nhãn, notebook chỉ báo chưa đủ dữ liệu, **không tạo metric giả**.

In [ ]:
reviewed=pd.read_csv(review_path)
reviewed['relevance']=pd.to_numeric(reviewed['relevance'],errors='coerce')
valid=reviewed.dropna(subset=['relevance']).copy()
if valid.empty:
    print('Chưa có nhãn thủ công: chưa thể tính Precision@K / nDCG@K.')
else:
    assert valid.relevance.isin([0,1,2]).all(),'relevance must be 0,1,2'
    metrics=[]
    for q,g in reviewed.groupby('query_id',sort=False):
        # Chỉ tính metric khi top 10 được chấm đủ theo thứ tự xuất file.
        top=g.head(10)
        if len(top)<10 or top.relevance.isna().any():
            print(q,': chưa chấm đủ top 10; bỏ qua');continue
        r=top.relevance.to_numpy(dtype=float)
        dcg=np.sum((2**r-1)/np.log2(np.arange(2,len(r)+2)))
        ideal=np.sort(r)[::-1]
        idcg=np.sum((2**ideal-1)/np.log2(np.arange(2,len(r)+2)))
        metrics.append({'query_id':q,'precision_at_10_rel_ge_1':float((r>=1).mean()),'ndcg_at_10_within_judged_top10':float(dcg/idcg) if idcg else 0.0})
    if metrics: display(pd.DataFrame(metrics))
    print('NOTE: nDCG computed only within judged top 10, not full-corpus nDCG.')

## 5. Xuất kết quả truy xuất & thống kê

In [ ]:
stats_path=REPORTS/'retrieval_test_summary.csv';pd.DataFrame(stats).to_csv(stats_path,index=False)
for name,frame in results.items():
    frame.to_csv(REPORTS/f'top50_{name}.csv',index=False,encoding='utf-8-sig')
print('Saved:',stats_path,'and top50_*.csv')
print('Pass hard filters + schema for',len(stats),'test queries. Relevance still requires manual judgments.')